# Inspect shard provenance

Each `shard_NNNNN.bin` in `token_shards_merged/` is a concatenation of many small per-file shards from the upstream tokenizer; the mapping is stored in `shard_manifest.json` under each entry's `merged_from` field. This notebook lets you look at any merged shard and see which original `.txt` files fed it (and vice versa).

Run cells in order. The first cell mounts Drive only if needed (it's fine to skip when running on a non-Colab box).

In [ ]:
import os, json

try:
    import google.colab  # noqa
    if not os.path.isdir('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
except ImportError:
    pass

# Prefer the local staged copy if a training run staged shards there;
# fall back to the canonical Drive copy.
CANDIDATES = [
    '/content/shards/shard_manifest.json',
    '/content/drive/MyDrive/synapse/token_shards_merged/shard_manifest.json',
]
MANIFEST_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if MANIFEST_PATH is None:
    raise FileNotFoundError(f'No manifest found at any of: {CANDIDATES}')

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
shards = manifest['shards']
by_name = {s['shard']: s for s in shards}
print(f'Loaded {len(shards)} shards from {MANIFEST_PATH}')

## Look up one shard → list source `.txt` files

In [ ]:
TARGET_SHARD = 'shard_02456.bin'   # change this

s = by_name.get(TARGET_SHARD)
if s is None:
    raise KeyError(f'{TARGET_SHARD} not in manifest')

sources = s.get('merged_from', [])
print(f"{s['shard']}  ({s.get('domain', '?')})  "
      f"{s['tokens']:,} tokens  |  {len(sources)} source files\n")

# merged_from is in concatenation order, so byte offsets are cumulative.
# uint16 shards = 2 bytes per token.
byte_off = 0
for src in sources:
    n = src['tokens']
    print(f"  bytes [{byte_off:>13,} .. {byte_off + n*2:>13,})  "
          f"{n:>10,} tok  {src['source']}")
    byte_off += n * 2

## Reverse lookup: find shards containing a given source `.txt`

In [ ]:
QUERY = 'finemath'   # substring matched against each merged_from source path

hits = []
for s in shards:
    matched = [src for src in s.get('merged_from', []) if QUERY in src['source']]
    if matched:
        hits.append((s['shard'], matched))

print(f"{len(hits)} shards contain a source matching {QUERY!r}\n")
for shard_name, matched in hits[:20]:
    print(f"  {shard_name}")
    for src in matched[:3]:
        print(f"      {src['tokens']:>10,} tok  {src['source']}")
    if len(matched) > 3:
        print(f'      ...and {len(matched)-3} more in this shard')
if len(hits) > 20:
    print(f'\n...and {len(hits)-20} more shards')

## Domain rollup: how many shards / tokens per source

Same numbers the trainer prints, but on the full manifest rather than the filtered set.

In [ ]:
from collections import defaultdict

rollup = defaultdict(lambda: {'shards': 0, 'tokens': 0, 'sources': 0})
for s in shards:
    dom = s.get('domain') or 'other'
    rollup[dom]['shards']  += 1
    rollup[dom]['tokens']  += s['tokens']
    rollup[dom]['sources'] += len(s.get('merged_from', []))

for dom, info in sorted(rollup.items(), key=lambda kv: -kv[1]['tokens']):
    print(f"  {dom:25s}  {info['shards']:>5}  shards  "
          f"{info['tokens']:>15,} tok  "
          f"{info['sources']:>6,} source files")